# Interactive Terrain Exploration in Jupyter

This notebook demonstrates rtxpy's `explore()` viewer running directly inside Jupyter.
The same GPU-accelerated ray tracing engine that powers the GLFW desktop viewer now
streams frames into an `ipywidgets.Image` widget with full mouse and keyboard controls.

**Requirements:**
```
pip install rtxpy[notebook] xarray rioxarray xrspatial
```

**Controls** (click the image first to focus):
- **W/A/S/D** or arrow keys: Move camera
- **Q/E**: Move up/down
- **I/J/K/L**: Look around
- **Click + drag**: Pan (slippy-map style)
- **Scroll wheel**: Zoom (FOV)
- **C**: Cycle colormap
- **T**: Toggle shadows
- **G**: Cycle terrain layers
- **M**: Toggle minimap
- **H**: Toggle help overlay
- **+/-**: Adjust movement speed

## 1. Load Terrain Data

We'll download SRTM elevation data for Crater Lake, Oregon — a collapsed volcanic
caldera with dramatic relief that looks great in 3D.

In [1]:
import numpy as np
import xarray as xr
from pathlib import Path

import rtxpy
from rtxpy import fetch_dem

print(f"rtxpy version: {rtxpy.__version__}")

rtxpy version: 0.0.9


In [2]:
BOUNDS = (-122.3, 42.8, -121.9, 43.0)  # Crater Lake, OR
CRS = 'EPSG:5070'  # NAD83 / Conus Albers
CACHE = Path.cwd()

terrain = fetch_dem(
    bounds=BOUNDS,
    output_path=CACHE / 'crater_lake_national_park.zarr',
    source='srtm',
    crs=CRS,
)

# Subsample for faster interactive performance
terrain = terrain[::2, ::2]
terrain.data = np.ascontiguousarray(terrain.data)

# Get stats before GPU transfer (cupy arrays don't allow implicit float())
elev_min = float(terrain.min())
elev_max = float(terrain.max())

# Transfer to GPU
terrain = terrain.rtx.to_cupy()

print(f"Shape: {terrain.shape}")
print(f"Elevation: {elev_min:.0f}m to {elev_max:.0f}m")

Using cached DEM: crater_lake_national_park.zarr
Shape: (620, 763)
Elevation: 1118m to 2710m


## 2. Basic Interactive Viewer

Call `explore()` just like you would from a script. In Jupyter it automatically
returns a widget instead of opening a GLFW window.

## 3. Multi-Layer Dataset

Build a Dataset with derived layers (slope, aspect, quantile). Press **G** to cycle
which layer drives the terrain coloring.

In [3]:
from xrspatial import slope, aspect, quantile

ds = xr.Dataset({
    'elevation': terrain.rename(None),
    'slope': slope(terrain),
    'aspect': aspect(terrain),
    'quantile': quantile(terrain),
})
print(ds)

<xarray.Dataset> Size: 9MB
Dimensions:      (x: 763, y: 620)
Coordinates:
  * x            (x) float64 6kB -2.112e+06 -2.112e+06 ... -2.074e+06 -2.074e+06
  * y            (y) float64 5kB 2.516e+06 2.516e+06 ... 2.486e+06 2.486e+06
    spatial_ref  int64 8B 0
Data variables:
    elevation    (y, x) float64 4MB array([[nan, nan, nan, ..., nan, nan, nan...
    slope        (y, x) float32 2MB array([[nan, nan, nan, ..., nan, nan, nan...
    aspect       (y, x) float32 2MB array([[nan, nan, nan, ..., nan, nan, nan...
    quantile     (y, x) float32 2MB array([[nan, nan, nan, ..., nan, nan, nan...


In [4]:
widget = ds.rtx.explore(
    z='elevation',
    width=1024,
    height=768,
    color_stretch='cbrt',
)

Overlay layers: slope, aspect, quantile  (press G to cycle)
OptiX 9.1.0 | RT Core 20 | NVIDIA RTX A6000 (sm_86) | Driver 591.44
rtxpy viewer (1024x768) — click image to focus, H for help, widget.stop() to exit


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x05\x03\x0…

In [ ]:
widget.stop()

## 4. Satellite Tiles + 3D Features

Drape satellite imagery on the terrain and place GeoJSON landmarks.
Press **U** to toggle the satellite overlay, **N** to cycle geometry groups.

In [5]:
import warnings

# Satellite tiles
ds.rtx.place_tiles('satellite', z='elevation')

# GeoJSON landmarks around Crater Lake
crater_lake_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.163, 42.947]},
            "properties": {"name": "Wizard Island"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.069, 42.922]},
            "properties": {"name": "Phantom Ship"},
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-122.142, 42.912]},
            "properties": {"name": "Rim Village"},
        },
        {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [-122.142, 42.912], [-122.160, 42.918],
                    [-122.175, 42.932], [-122.178, 42.948],
                    [-122.168, 42.962], [-122.148, 42.976],
                    [-122.118, 42.982], [-122.088, 42.978],
                    [-122.065, 42.968], [-122.045, 42.952],
                    [-122.035, 42.935], [-122.040, 42.918],
                    [-122.055, 42.905], [-122.080, 42.897],
                    [-122.110, 42.897], [-122.135, 42.903],
                    [-122.142, 42.912],
                ],
            },
            "properties": {"name": "Rim Drive"},
        },
    ],
}

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='place_geojson called before')
    info = ds.rtx.place_geojson(
        crater_lake_geojson,
        z='elevation',
        height=15.0,
        label_field='name',
        geometry_id='landmark',
    )
print(f"Placed {info['geometries']} GeoJSON features")

  Tile service: zoom 13, bounds (42.7242, -122.3727) - (43.0760, -121.8290)
  Fetching 156 tiles at zoom 13...
Placed 4 GeoJSON features


In [ ]:
from rtxpy import fetch_wind

wind = fetch_wind(BOUNDS, grid_size=15)
wind['n_particles'] = 15000
wind['max_age'] = 120
wind['speed_mult'] = 400.0

print(f"Wind grid: {wind['u'].shape}, "
      f"mean speed: {np.sqrt(wind['u']**2 + wind['v']**2).mean():.1f} m/s")

In [ ]:
widget = ds.rtx.explore(
    z='elevation',
    width=1024,
    height=768,
    mesh_type='voxel',
    color_stretch='cbrt',
    wind_data=wind,
)

In [ ]:
widget.stop()

## 6. Advanced: AO + Denoiser

Enable ambient occlusion and the OptiX AI denoiser for higher quality rendering.
AO accumulates progressively — the image gets cleaner over time. Toggle at runtime
with **0** (AO) and **Shift+D** (denoiser).

In [ ]:
widget = ds.rtx.explore(
    z='elevation',
    width=1024,
    height=768,
    mesh_type='voxel',
    color_stretch='cbrt',
    ao_samples=1,
    denoise=True,
)

In [ ]:
widget.stop()

## 7. Custom Camera Start Position

Set a specific camera position and look-at point to start from a particular viewpoint.
Coordinates are in world units (meters in the projected CRS).

In [ ]:
# Get terrain extents for positioning
H, W = terrain.shape
spacing_x = terrain.rtx._pixel_spacing_x
spacing_y = terrain.rtx._pixel_spacing_y
world_W = W * spacing_x
world_H = H * spacing_y

# Start looking down into the caldera from the rim
widget = terrain.rtx.explore(
    width=1024,
    height=768,
    start_position=(world_W * 0.5, world_H * 0.8, elev_max + 500),
    look_at=(world_W * 0.5, world_H * 0.45, elev_min),
)

In [ ]:
widget.stop()